In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
titanic = sns.load_dataset("titanic")

In [3]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [5]:
print("Shape:", titanic.shape)
print("Columns:", titanic.columns.tolist())

Shape: (891, 15)
Columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


In [6]:
df = titanic[['sex', 'class', 'embark_town', 'alone', 'survived']].dropna()

In [7]:
df.shape

(889, 5)

In [ ]:
# columnTransformer
# Pipeline

In [8]:
df.head()

,sex,class,embark_town,alone,survived
0,male,Third,Southampton,False,0
1,female,First,Cherbourg,False,1
2,female,Third,Southampton,True,1
3,female,First,Southampton,False,1
4,male,Third,Southampton,True,0


In [9]:
label_encoders = {}
for col in ['sex', 'class', 'embark_town', 'alone']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [10]:
label_encoders

{'sex': LabelEncoder(),
 'class': LabelEncoder(),
 'embark_town': LabelEncoder(),
 'alone': LabelEncoder()}

In [11]:
X = df.drop('survived', axis=1)
y = df['survived']

In [12]:
X.head(2)

,sex,class,embark_town,alone
0,1,2,2,0
1,0,0,0,0


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10, stratify=y
)

In [15]:
cat_nb = CategoricalNB()

In [16]:
cat_nb.fit(X_train, y_train)

,"alpha alpha: float, default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"min_categories min_categories: int or array-like of shape (n_features,), default=NoneMinimum number of categories per feature.- integer: Sets the minimum number of categories per feature to `n_categories` for each features.- array-like: shape (n_features,) where `n_categories[i]` holds the minimum number of categories for the ith column of the input.- None (default): Determines the number of categories automatically from the training data... versionadded:: 0.24",None


In [17]:
y_train_pred = cat_nb.predict(X_train)
y_test_pred = cat_nb.predict(X_test)

In [18]:
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))

Train Accuracy: 0.7609001406469761
Test Accuracy: 0.7696629213483146


In [19]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

Confusion Matrix:
 [[93 17]
 [24 44]]


In [20]:
cv_scores = cross_val_score(cat_nb, X_train, y_train, cv=5, scoring='accuracy')

In [21]:
cv_scores

array([0.76223776, 0.71126761, 0.75352113, 0.79577465, 0.74647887])

In [22]:
print("Cross-Validation Accuracy (5-fold):", np.mean(cv_scores).round(4))

Cross-Validation Accuracy (5-fold): 0.7539


# Prediction on sample

In [23]:
sex_cnb = "female"
class_cnb = "First"
embark_town_cnb = "Southampton"
alone_cnb = False

In [24]:
label_encoders

{'sex': LabelEncoder(),
 'class': LabelEncoder(),
 'embark_town': LabelEncoder(),
 'alone': LabelEncoder()}

In [25]:
sample = pd.DataFrame({
    'sex': [label_encoders['sex'].transform([sex_cnb])[0]],
    'class': [label_encoders['class'].transform([class_cnb])[0]],
    'embark_town': [label_encoders['embark_town'].transform([embark_town_cnb])[0]],
    'alone': [label_encoders['alone'].transform([alone_cnb])[0]]
})

In [26]:
prediction = cat_nb.predict(sample)
prediction

array([1])

In [27]:
prediction = cat_nb.predict(sample)[0]
prediction

np.int64(1)

In [28]:
sample

,sex,class,embark_town,alone
0,0,0,2,0


In [29]:
print("Predicted Survival:", "Survived" if prediction == 1 else "Did not survive")

Predicted Survival: Survived
